# PBMC 3k Single-Cell RNA-seq Analysis

Standard workflow: QC, normalization, clustering, marker genes, and cell type annotation.

In [ ]:
import scanpy as sc
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

sc.settings.verbosity = 1
sc.settings.figdir = '../results/'
sc.settings.set_figure_params(dpi=100, frameon=False)

## Load data

In [ ]:
adata = sc.datasets.pbmc3k()
adata.var_names_make_unique()
print(f"{adata.n_obs} cells x {adata.n_vars} genes")

## QC

In [ ]:
adata.var['mt'] = adata.var_names.str.startswith('MT-')
sc.pp.calculate_qc_metrics(adata, qc_vars=['mt'], percent_top=None, log1p=False, inplace=True)

sc.pl.violin(adata, ['n_genes_by_counts', 'total_counts', 'pct_counts_mt'], jitter=0.4, multi_panel=True, save='_qc_violin.pdf')

In [ ]:
sc.pl.scatter(adata, x='total_counts', y='pct_counts_mt', save='_mt_scatter.pdf')
sc.pl.scatter(adata, x='total_counts', y='n_genes_by_counts', save='_genes_scatter.pdf')

In [ ]:
# Filter cells and genes
adata = adata[adata.obs.n_genes_by_counts < 2500, :]
adata = adata[adata.obs.pct_counts_mt < 5, :]
sc.pp.filter_genes(adata, min_cells=3)

print(f"After filtering: {adata.n_obs} cells x {adata.n_vars} genes")

## Normalization and HVG selection

In [ ]:
adata.raw = adata.copy()

sc.pp.normalize_total(adata, target_sum=1e4)
sc.pp.log1p(adata)

sc.pp.highly_variable_genes(adata, min_mean=0.0125, max_mean=3, min_disp=0.5)
print(f"{adata.var.highly_variable.sum()} highly variable genes")

sc.pl.highly_variable_genes(adata, save='_hvg.pdf')

In [ ]:
adata = adata[:, adata.var.highly_variable]
sc.pp.regress_out(adata, ['total_counts', 'pct_counts_mt'])
sc.pp.scale(adata, max_value=10)

## PCA and neighborhood graph

In [ ]:
sc.tl.pca(adata, svd_solver='arpack')
sc.pl.pca_variance_ratio(adata, log=True, n_pcs=50, save='_pca_variance.pdf')

In [ ]:
sc.pp.neighbors(adata, n_neighbors=10, n_pcs=40)
sc.tl.umap(adata)

## Clustering

In [ ]:
sc.tl.leiden(adata, resolution=0.9)
sc.pl.umap(adata, color='leiden', save='_leiden_clusters.pdf')

## Marker genes

In [ ]:
sc.tl.rank_genes_groups(adata, 'leiden', method='wilcoxon')
sc.pl.rank_genes_groups(adata, n_genes=10, sharey=False, save='_markers.pdf')

In [ ]:
# Check known PBMC markers
marker_genes = {
    'CD4 T': ['IL7R', 'CD4'],
    'CD8 T': ['CD8A', 'CD8B'],
    'NK': ['GNLY', 'NKG7'],
    'B': ['MS4A1', 'CD79A'],
    'Monocytes': ['CD14', 'LYZ', 'FCGR3A'],
    'Dendritic': ['FCER1A', 'CST3'],
    'Platelets': ['PPBP']
}

sc.pl.dotplot(adata, var_names={k: v for k, v in marker_genes.items()}, groupby='leiden', save='_dotplot_markers.pdf')

## Cell type annotation

In [ ]:
# Assign based on marker expression per cluster
# Mapping determined by inspecting dotplot and top marker genes above
cluster_to_celltype = {
    '0': 'CD4 T',
    '1': 'CD14 Monocytes',
    '2': 'B cells',
    '3': 'CD8 T',
    '4': 'FCGR3A Monocytes',
    '5': 'NK',
    '6': 'Dendritic',
    '7': 'Platelets'
}

adata.obs['cell_type'] = adata.obs['leiden'].map(cluster_to_celltype)
sc.pl.umap(adata, color='cell_type', save='_cell_types.pdf')

## Differential expression: CD14 vs FCGR3A Monocytes

In [ ]:
monocytes = adata[adata.obs['cell_type'].isin(['CD14 Monocytes', 'FCGR3A Monocytes'])].copy()
sc.tl.rank_genes_groups(monocytes, 'cell_type', method='wilcoxon')
sc.pl.rank_genes_groups(monocytes, n_genes=15, save='_de_monocytes.pdf')

de_results = sc.get.rank_genes_groups_df(monocytes, group='CD14 Monocytes')
de_results.head(20)

In [ ]:
adata.write('../results/pbmc3k_analyzed.h5ad')
print('Done.')